# MAPED: merge a tilt series into one 4D-STEM dataset

MAPED records the same area at several beam tilts, aligns the scans, and averages them into one dataset. Edit the two paths in the first cell, then run the cells in order.

**GPU.** The same code runs on an NVIDIA GPU (CUDA) or on an Apple Silicon Mac (MPS); nothing to set. On a workstation with several NVIDIA GPUs the first one is used; to pick another, pass `device="cuda:1"` to `io.load`. On a Mac there is one GPU, `mps`. The next cell lists what this machine has.


In [ ]:
# Install once. The widget brings quantem.gpu with it; MAPED lives in quantem.
# pip install --extra-index-url https://test.pypi.org/simple "quantem-gpu>=0.0.1rc11" git+https://github.com/bobleesj/quantem.widget.git
# pip install git+https://github.com/bobleesj/quantem.git@main

In [ ]:
from quantem.gpu import io, device
from quantem.diffraction import MAPED
from quantem.widget import Show4DSTEM

DATA_DIR = "/path/to/sample/tilts"
SAVE_PATH = "/path/to/results/merged_master.h5"

files = io.discover(DATA_DIR)

## GPU

The first device listed is used.


In [ ]:
device.profile()["available_devices"]

## Load

Each tilt is read once and kept compressed on the GPU. Detector-flagged bad pixels are replaced by the median of their neighbors; the files on disk are unchanged (`hot_pixel_correction="none"` keeps raw values).

In [ ]:
tilts = io.load(files)

## Merge

One call: beam centers, detector alignment, scan alignment, merge. One line per stage; four figures. The result is cropped to the scan positions every tilt covers and stored as scaled 16-bit (rounding error printed). Settings: `maped.run(iterations=80)`, `maped.run(show=False)` for scripts.

In [ ]:
maped = MAPED(tilts, files)  # file names carry the tilt, used in plot titles
merged = maped.run()

## Save, load back, view

The saved file reopens with the same values; the viewer shows what is on disk.

In [ ]:
io.save(SAVE_PATH, merged)
maped.close()  # release the merged result on the GPU before loading the file
saved = io.load(SAVE_PATH)

In [ ]:
Show4DSTEM(saved)

## Selected diffraction patterns

Three scan positions, read from the saved file; indexing returns a Torch tensor.


In [ ]:
from quantem.core.visualization import show_2d

positions = [(int(saved.shape[0] * f), int(saved.shape[1] * f)) for f in (0.25, 0.5, 0.75)]
fig, ax = show_2d([saved[row, col] for row, col in positions], title=[f"scan ({row}, {col})" for row, col in positions], cmap="inferno", norm="power_sqrt")

## See also

- [maped_load.ipynb](maped_load.ipynb): open a saved result later.
- [maped_step_by_step.ipynb](maped_step_by_step.ipynb): the stages one at a time, with what each setting changes.
- Exact float32 for a patch: `maped.merge(scan_region=(252, 260, 252, 260))`.
- Small-memory Mac: `maped.run(save_to=SAVE_PATH, release_tilts=True)` streams the result to disk.